In [1]:
%matplotlib inline
import sys
sys.path.append("..")
import typer
import rich
import numpy as np
import pandas as pd
import uproot
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
from pathlib import Path

import gp
from gp import mass_resolution
from gp import GaussianProcessModel
from gp import kernels
from gp import _hist
from gp import mass_resolution
from gp._plot import plt, label

the_kernel = (
    gp.kernels.WhiteKernel(noise_level=7e3) +
    gp.kernels.RBF(length_scale=0.016) * gp.kernels.DotProduct(sigma_0=2.5e4)
)

rebin = 2
"""
the_kernel = (
    gp.kernels.WhiteKernel() +
    gp.kernels.RBF() * gp.kernels.DotProduct()
)
"""
def search(
    output: Path,
    mass_range: list = [0.160],
    blind_halfwidth: float = 1.96,
    plot_each: bool = False,
    input_file: str = "invariant_mass_0pt5mm_full.root",
    hist_name: str = "invariant_mass"
):
    """Search through mass points, performing a fit at each one"""
    
    # Determine start, stop, and step from mass_range
    start, stop, step = 0.040, None, 0.005
    if len(mass_range) == 1:
        stop = mass_range[0]
    elif len(mass_range) == 2:
        start, stop = mass_range
    elif len(mass_range) == 3:
        start, stop, step = mass_range
    else:
        raise ValueError("mass_range should have 1 to 3 values")

    # Create output directory if not existing
    output.mkdir(exist_ok=True, parents=True) 
    
    # Load the histogram using uproot
    with uproot.open(input_file) as f:
        h = f[hist_name].to_hist()  # Adjust if this is not correct; you might need to convert it to a format compatible with your model

    mass_points = np.arange(start, stop, step)
    results = []

    for mass, sigma_m in tqdm(zip(mass_points, mass_resolution(mass_points)), total=len(mass_points)):
        # Initialize GaussianProcessModel with histogram and kernel
        gpm = GaussianProcessModel(
            h=h,
            kernel=the_kernel,
            blind_range=(mass - blind_halfwidth * sigma_m, mass + blind_halfwidth * sigma_m),
            modify_histogram= _hist.manipulation.rebin_and_limit(rebin),
            #modify_histogram=[_hist.manipulation.rebin_and_limit(10), _hist.manipulation.inject_signal(5000, mass - blind_halfwidth * sigma_m, .036)],
            empty_bin_variance=3.688
        )

        if plot_each:
            fig, axes = gpm.plot_comparison(l=
            '\n'.join([
                'GP with 95% CI',
                f'$\chi^2 = ${gpm.chi2_statistic:.3g}',
                r'$P_{\chi^2} = $'+f'{gpm.p_value:.3g}',
                'Blind Range: ' + str(round(gpm.blind_range[1]-gpm.blind_range[0], 5))
                #'Signal Injected: ' + str(events)
            ]))
            fig.savefig(output / f'{int(1000 * mass)}mev_search.png', bbox_inches='tight')
            plt.close()
        
        prediction, uncertainty = gpm.blind_single_bin_prediction
        observation = gpm.blind_single_bin_observation
        results.append([
            mass, sigma_m, np.sum(gpm.pull**2), prediction, uncertainty, observation,
            gpm.kernel.k1.noise_level, gpm.kernel.k2.k1.length_scale, gpm.kernel.k2.k2.sigma_0
        ])

    # Write results to CSV
    results_df = pd.DataFrame(results, columns=[
        'mass', 'sigma_m', 'chi2', 'prediction', 'uncertainty', 'observation', 
        'noise_level', 'length_scale', 'sigma_0'
    ])
    results_df.to_csv(output / 'search-results.csv', index=False)

    # Filter good results and plot summary
    good_results = results_df[results_df['chi2'] > 200]
    fig, axes = plt.subplots(nrows=3, sharex='col', gridspec_kw={'hspace': 0.05})
    axes[0].annotate(
        r'$K(m_i, m_j) = (\sigma_0^2 + m_i m_j \delta_{ij})e^{-(m_i-m_j)^2/\ell^2}$',
        (0.5, 0.9), xycoords='axes fraction', va='top', ha='center'
    )

    for ax, column, ylabel in zip(
        axes, ['chi2', 'length_scale', 'sigma_0'], 
        [r'$\chi^2$', r'$\ell$ / GeV', r'$\sigma_0$ / GeV']
    ):
        ax.plot(good_results['mass'], good_results[column])
        ax.set_ylabel(ylabel)
    
    axes[-1].set_xlabel('mass / GeV')
    label(ax=axes[0])
    fig.savefig(output / 'search-good-fit-overview.png', bbox_inches='tight')
    plt.close()


<>:80: SyntaxWarning: invalid escape sequence '\c'
<>:80: SyntaxWarning: invalid escape sequence '\c'
/var/folders/lh/5k1xk47s7q9dgdbf5kby_6_h0000gn/T/ipykernel_1647/3240122147.py:80: SyntaxWarning: invalid escape sequence '\c'
  f'$\chi^2 = ${gpm.chi2_statistic:.3g}',


In [2]:
search(
    output=Path("outputs_2015_rbn2_alpha_blind_halfwidth_1"),  # Replace with desired output path
    mass_range=[0.015, 0.120, 0.001],
    blind_halfwidth=1,
    plot_each=True,
    input_file="invariant_mass_0pt5mm_full.root",
    hist_name="invariant_mass"
)


  1%|▍                                          | 1/105 [00:34<59:28, 34.32s/it]
/var/folders/lh/5k1xk47s7q9dgdbf5kby_6_h0000gn/T/ipykernel_1647/3240122147.py:80: SyntaxWarning: invalid escape sequence '\c'
  f'$\chi^2 = ${gpm.chi2_statistic:.3g}',
/Users/aidanhsu/Documents/gaus-proc/notebooks/../gp/__init__.py:198: SyntaxWarning: invalid escape sequence '\c'
  f'$\chi^2 = ${self.chi2_statistic:.3g}',


KeyboardInterrupt: 

In [ ]:
print(sys.executable)